# IDF Document to PDF Generator

This notebook generates a professional PDF document from the Invention Disclosure Form (IDFb.md) with all figures and formatting preserved.

## 1. Install Required Packages

In [5]:
# Install required packages (fpdf2 works without system dependencies)
%pip install markdown fpdf2 Pillow -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import Libraries

In [1]:
import os
import markdown
from fpdf import FPDF
from datetime import datetime
import re
from pathlib import Path
from PIL import Image

## 3. Configuration

In [2]:
# File paths
MARKDOWN_FILE = 'IDFb.md'
OUTPUT_PDF = 'IDF_Document.pdf'
BASE_DIR = Path('.').resolve()

print(f"Base Directory: {BASE_DIR}")
print(f"Input File: {MARKDOWN_FILE}")
print(f"Output PDF: {OUTPUT_PDF}")

Base Directory: D:\Number Spoofing Patent
Input File: IDFb.md
Output PDF: IDF_Document.pdf


## 4. Define CSS Styles for PDF

In [3]:
# PDF generation class with custom styling
class IDFDocument(FPDF):
    def __init__(self):
        super().__init__()
        self.set_auto_page_break(auto=True, margin=20)
        
    def header(self):
        if self.page_no() > 1:
            self.set_font('Helvetica', 'I', 8)
            self.set_text_color(100, 100, 100)
            self.cell(0, 10, 'AI-Based Multi-Layer Spoofing Detection System - IDF', 0, 0, 'C')
            self.ln(15)
    
    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.set_text_color(100, 100, 100)
        self.cell(0, 10, f'Page {self.page_no()}/{{nb}}', 0, 0, 'C')
        self.cell(0, 10, 'CONFIDENTIAL', 0, 0, 'R')
    
    def chapter_title(self, title, level=1):
        if level == 1:
            self.set_font('Helvetica', 'B', 18)
            self.set_text_color(26, 54, 93)
        elif level == 2:
            self.set_font('Helvetica', 'B', 14)
            self.set_text_color(30, 64, 175)
        elif level == 3:
            self.set_font('Helvetica', 'B', 12)
            self.set_text_color(30, 58, 138)
        else:
            self.set_font('Helvetica', 'B', 11)
            self.set_text_color(55, 65, 81)
        
        self.multi_cell(0, 8, title)
        self.ln(4)
        self.set_text_color(0, 0, 0)
    
    def body_text(self, text):
        self.set_font('Helvetica', '', 10)
        self.set_text_color(51, 51, 51)
        self.multi_cell(0, 5, text)
        self.ln(2)
    
    def add_image_with_caption(self, img_path, caption, max_width=170):
        if os.path.exists(img_path):
            try:
                # Get image dimensions
                with Image.open(img_path) as img:
                    w, h = img.size
                    aspect = h / w
                
                # Calculate display size
                display_width = min(max_width, 170)
                display_height = display_width * aspect
                
                # Check if we need a new page
                if self.get_y() + display_height + 20 > 270:
                    self.add_page()
                
                # Center the image
                x = (210 - display_width) / 2
                self.image(img_path, x=x, w=display_width)
                self.ln(3)
                
                # Add caption
                self.set_font('Helvetica', 'I', 9)
                self.set_text_color(75, 85, 99)
                self.multi_cell(0, 4, caption, align='C')
                self.ln(5)
                self.set_text_color(0, 0, 0)
            except Exception as e:
                self.body_text(f"[Image: {caption} - Error loading: {e}]")
        else:
            self.set_font('Helvetica', 'I', 9)
            self.set_text_color(150, 150, 150)
            self.multi_cell(0, 5, f"[Image not found: {img_path}]", align='C')
            self.set_font('Helvetica', 'I', 9)
            self.multi_cell(0, 4, caption, align='C')
            self.ln(3)
            self.set_text_color(0, 0, 0)
    
    def add_table(self, headers, rows):
        self.set_font('Helvetica', 'B', 9)
        self.set_fill_color(30, 64, 175)
        self.set_text_color(255, 255, 255)
        
        # Calculate column widths
        num_cols = len(headers)
        col_width = 180 / num_cols
        
        # Header row
        for header in headers:
            self.cell(col_width, 7, header[:20], 1, 0, 'C', True)
        self.ln()
        
        # Data rows
        self.set_font('Helvetica', '', 8)
        self.set_text_color(0, 0, 0)
        fill = False
        for row in rows:
            if fill:
                self.set_fill_color(243, 244, 246)
            else:
                self.set_fill_color(255, 255, 255)
            for cell in row:
                self.cell(col_width, 6, str(cell)[:25], 1, 0, 'L', True)
            self.ln()
            fill = not fill
        self.ln(5)

print("PDF document class defined successfully!")

PDF document class defined successfully!


## 5. Read and Process Markdown File

In [4]:
def read_markdown_file(filepath):
    """Read the markdown file content."""
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    return content

def parse_markdown_content(content):
    """Parse markdown content into structured elements."""
    lines = content.split('\n')
    elements = []
    
    i = 0
    while i < len(lines):
        line = lines[i]
        
        # Headers
        if line.startswith('#### '):
            elements.append(('h4', line[5:].strip()))
        elif line.startswith('### '):
            elements.append(('h3', line[4:].strip()))
        elif line.startswith('## '):
            elements.append(('h2', line[3:].strip()))
        elif line.startswith('# '):
            elements.append(('h1', line[2:].strip()))
        
        # Images
        elif line.startswith('!['):
            match = re.match(r'!\[([^\]]*)\]\(([^)]+)\)', line)
            if match:
                alt_text = match.group(1)
                img_path = match.group(2)
                elements.append(('image', img_path, alt_text))
        
        # Figure captions (italics after images)
        elif line.startswith('*Figure') or line.startswith('*'):
            caption = line.strip('*').strip()
            elements.append(('caption', caption))
        
        # Tables
        elif '|' in line and i + 1 < len(lines) and '---' in lines[i + 1]:
            # Parse table
            headers = [h.strip() for h in line.split('|') if h.strip()]
            i += 1  # Skip separator
            rows = []
            i += 1
            while i < len(lines) and '|' in lines[i]:
                row = [c.strip() for c in lines[i].split('|') if c.strip()]
                if row:
                    rows.append(row)
                i += 1
            elements.append(('table', headers, rows))
            continue
        
        # Code blocks
        elif line.startswith('```'):
            code_lines = []
            i += 1
            while i < len(lines) and not lines[i].startswith('```'):
                code_lines.append(lines[i])
                i += 1
            elements.append(('code', '\\n'.join(code_lines)))
        
        # Horizontal rule
        elif line.strip() == '---':
            elements.append(('hr',))
        
        # Regular text
        elif line.strip():
            # Check for bullet points
            if line.strip().startswith('- ') or line.strip().startswith('* '):
                elements.append(('bullet', line.strip()[2:]))
            elif re.match(r'^\d+\. ', line.strip()):
                elements.append(('numbered', line.strip()))
            else:
                elements.append(('text', line.strip()))
        
        i += 1
    
    return elements

# Read and parse the markdown file
md_content = read_markdown_file(MARKDOWN_FILE)
print(f"Read {len(md_content)} characters from {MARKDOWN_FILE}")

elements = parse_markdown_content(md_content)
print(f"Parsed {len(elements)} elements from the document")

Read 26098 characters from IDFb.md
Parsed 283 elements from the document


## 6. Convert Markdown to HTML

In [11]:
def clean_text(text):
    """Remove or replace characters not supported by Helvetica."""
    # Replace common problematic characters
    replacements = {
        '\u2019': "'",  # Right single quote
        '\u2018': "'",  # Left single quote
        '\u201c': '"',  # Left double quote
        '\u201d': '"',  # Right double quote
        '\u2013': '-',  # En dash
        '\u2014': '-',  # Em dash
        '\u2022': '-',  # Bullet
        '\u2026': '...',  # Ellipsis
        '\u00a0': ' ',  # Non-breaking space
        '\u2502': '|',  # Box drawing
        '\u250c': '+',  # Box drawing
        '\u2510': '+',  # Box drawing
        '\u2514': '+',  # Box drawing
        '\u2518': '+',  # Box drawing
        '\u2500': '-',  # Box drawing
        '\u25bc': 'v',  # Down arrow
        '\u25b6': '>',  # Right arrow
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    # Remove any remaining non-ASCII characters except common ones
    return ''.join(c if ord(c) < 128 or c in 'éèêëàâäùûüôöîïç' else ' ' for c in text)

def generate_pdf(elements, output_path):
    """Generate PDF from parsed markdown elements."""
    pdf = IDFDocument()
    pdf.alias_nb_pages()
    pdf.set_left_margin(15)
    pdf.set_right_margin(15)
    pdf.add_page()
    
    # Title page
    pdf.set_font('Helvetica', 'B', 24)
    pdf.set_text_color(26, 54, 93)
    pdf.ln(40)
    pdf.multi_cell(0, 12, 'Invention Disclosure Form (IDF)', align='C')
    pdf.ln(10)
    pdf.set_font('Helvetica', 'B', 14)
    pdf.set_text_color(30, 64, 175)
    pdf.multi_cell(0, 8, 'AI-Based Multi-Layer Spoofing Detection System', align='C')
    pdf.ln(3)
    pdf.multi_cell(0, 8, 'Using Deep Learning and Adaptive Policy Networks', align='C')
    pdf.ln(3)
    pdf.multi_cell(0, 8, 'for Real-Time Network Security', align='C')
    pdf.ln(30)
    pdf.set_font('Helvetica', '', 11)
    pdf.set_text_color(100, 100, 100)
    pdf.multi_cell(0, 6, f'Generated: {datetime.now().strftime("%B %d, %Y")}', align='C')
    pdf.ln(5)
    pdf.multi_cell(0, 6, 'CONFIDENTIAL - For Internal Use Only', align='C')
    
    # Start content
    pdf.add_page()
    
    skip_next_caption = False
    
    for elem in elements:
        elem_type = elem[0]
        
        try:
            if elem_type == 'h1':
                pdf.add_page()
                pdf.chapter_title(clean_text(elem[1]), 1)
                pdf.ln(2)
                pdf.set_draw_color(37, 99, 235)
                pdf.line(15, pdf.get_y(), 195, pdf.get_y())
                pdf.ln(5)
            
            elif elem_type == 'h2':
                if pdf.get_y() > 240:
                    pdf.add_page()
                pdf.ln(3)
                pdf.chapter_title(clean_text(elem[1]), 2)
            
            elif elem_type == 'h3':
                if pdf.get_y() > 250:
                    pdf.add_page()
                pdf.chapter_title(clean_text(elem[1]), 3)
            
            elif elem_type == 'h4':
                pdf.chapter_title(clean_text(elem[1]), 4)
            
            elif elem_type == 'image':
                img_path = elem[1]
                alt_text = elem[2] if len(elem) > 2 else ''
                pdf.add_image_with_caption(img_path, clean_text(alt_text))
                skip_next_caption = True
            
            elif elem_type == 'caption':
                if not skip_next_caption:
                    caption_text = clean_text(elem[1])
                    if caption_text.strip():
                        pdf.set_font('Helvetica', 'I', 9)
                        pdf.set_text_color(75, 85, 99)
                        pdf.multi_cell(0, 4, caption_text, align='C')
                        pdf.ln(3)
                        pdf.set_text_color(0, 0, 0)
                skip_next_caption = False
            
            elif elem_type == 'table':
                headers = [clean_text(h) for h in elem[1]]
                rows = [[clean_text(str(c)) for c in row] for row in elem[2]]
                if pdf.get_y() > 220:
                    pdf.add_page()
                pdf.add_table(headers, rows)
            
            elif elem_type == 'code':
                if pdf.get_y() > 200:
                    pdf.add_page()
                pdf.set_font('Courier', '', 7)
                pdf.set_fill_color(240, 240, 240)
                pdf.set_text_color(50, 50, 50)
                code_text = clean_text(elem[1].replace('\\n', '\n'))
                pdf.multi_cell(0, 3.5, code_text, fill=True)
                pdf.ln(3)
                pdf.set_text_color(0, 0, 0)
            
            elif elem_type == 'bullet':
                pdf.set_font('Helvetica', '', 10)
                pdf.set_x(20)
                bullet_text = f"- {clean_text(elem[1])}"
                pdf.multi_cell(0, 5, bullet_text)
            
            elif elem_type == 'numbered':
                pdf.set_font('Helvetica', '', 10)
                pdf.set_x(20)
                pdf.multi_cell(0, 5, clean_text(elem[1]))
            
            elif elem_type == 'hr':
                pdf.ln(5)
                pdf.set_draw_color(200, 200, 200)
                pdf.line(15, pdf.get_y(), 195, pdf.get_y())
                pdf.ln(5)
            
            elif elem_type == 'text':
                text = clean_text(elem[1])
                if text.strip():
                    if text.startswith('**') and text.endswith('**'):
                        pdf.set_font('Helvetica', 'B', 10)
                        text = text[2:-2]
                    else:
                        pdf.set_font('Helvetica', '', 10)
                    pdf.set_text_color(51, 51, 51)
                    pdf.multi_cell(0, 5, text)
                    pdf.ln(1)
        except Exception as e:
            # Skip problematic elements
            print(f"Warning: Skipped element due to: {e}")
            continue
    
    # Save PDF
    pdf.output(output_path)
    return True

print("PDF generation function defined!")

PDF generation function defined!


## 7. Create Complete HTML Document

In [12]:
# Generate the PDF document
print("Generating PDF document...")
print("This may take a moment as images are being processed...")
print()

try:
    success = generate_pdf(elements, OUTPUT_PDF)
    
    if success and os.path.exists(OUTPUT_PDF):
        file_size = os.path.getsize(OUTPUT_PDF)
        print(f"✓ PDF generated successfully!")
        print(f"✓ Output: {OUTPUT_PDF}")
        print(f"✓ Size: {file_size / 1024:.1f} KB")
    else:
        print("✗ PDF generation failed")
        
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

Generating PDF document...
This may take a moment as images are being processed...



C:\Users\ASUS\AppData\Local\Temp\ipykernel_52380\3046664105.py:18: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.
  self.cell(0, 10, f'Page {self.page_no()}/{{nb}}', 0, 0, 'C')
C:\Users\ASUS\AppData\Local\Temp\ipykernel_52380\3046664105.py:19: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.
  self.cell(0, 10, 'CONFIDENTIAL', 0, 0, 'R')
C:\Users\ASUS\AppData\Local\Temp\ipykernel_52380\3046664105.py:11: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.
  self.cell(0, 10, 'AI-Based Multi-Layer Spoofing Detection System - IDF', 0, 0, 'C')
C:\Users\ASUS\AppData\Local\Temp\ipykernel_52380\3046664105.py:94: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.
  self.cell(col_width, 7, header[:20], 1, 0, 'C', Tru

✓ PDF generated successfully!
✓ Output: IDF_Document.pdf
✓ Size: 8215.4 KB


## 8. Generate PDF

In [13]:
# Verification and Summary
print("=" * 60)
print("               PDF GENERATION SUMMARY")
print("=" * 60)

if os.path.exists(OUTPUT_PDF):
    file_size = os.path.getsize(OUTPUT_PDF)
    mod_time = datetime.fromtimestamp(os.path.getmtime(OUTPUT_PDF))
    
    print(f"\n  Status: SUCCESS")
    print(f"\n  Output File: {OUTPUT_PDF}")
    print(f"  File Size: {file_size / 1024 / 1024:.2f} MB ({file_size:,} bytes)")
    print(f"  Generated: {mod_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\n  Source: {MARKDOWN_FILE}")
    print(f"  Working Directory: {BASE_DIR}")
    
    print("\n" + "-" * 60)
    print("Document Contents:")
    print("-" * 60)
    print("  - Title: AI-Based Multi-Layer Spoofing Detection System")
    print("  - Sections: 9 main sections + 2 appendices")
    print("  - Figures: 25 figures with captions")
    print("  - Tables: 10+ data tables")
    print("  - Patent Claims: 8 primary claims")
    
    print("\n" + "=" * 60)
    print(f"\n  Full path: {os.path.abspath(OUTPUT_PDF)}")
else:
    print(f"\n  Status: FAILED")
    print(f"  PDF file was not created.")

               PDF GENERATION SUMMARY

  Status: SUCCESS

  Output File: IDF_Document.pdf
  File Size: 8.02 MB (8,412,553 bytes)
  Generated: 2026-02-08 22:17:13

  Source: IDFb.md
  Working Directory: D:\Number Spoofing Patent

------------------------------------------------------------
Document Contents:
------------------------------------------------------------
  - Title: AI-Based Multi-Layer Spoofing Detection System
  - Sections: 9 main sections + 2 appendices
  - Figures: 25 figures with captions
  - Tables: 10+ data tables
  - Patent Claims: 8 primary claims


  Full path: d:\Number Spoofing Patent\IDF_Document.pdf


## 9. Verification and Summary

In [ ]:
def verify_output(pdf_path):
    """Verify the generated PDF and display summary."""
    
    print("="*60)
    print("               PDF GENERATION SUMMARY")
    print("="*60)
    
    if os.path.exists(pdf_path):
        file_size = os.path.getsize(pdf_path)
        mod_time = datetime.fromtimestamp(os.path.getmtime(pdf_path))
        
        print(f"\n✓ Status: SUCCESS")
        print(f"\n  Output File: {pdf_path}")
        print(f"  File Size: {file_size / 1024 / 1024:.2f} MB ({file_size:,} bytes)")
        print(f"  Generated: {mod_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"\n  Source: {MARKDOWN_FILE}")
        print(f"  Working Directory: {BASE_DIR}")
        
        print("\n" + "-"*60)
        print("Document Contents:")
        print("-"*60)
        print("  • Title: AI-Based Multi-Layer Spoofing Detection System")
        print("  • Sections: 9 main sections + 2 appendices")
        print("  • Figures: 25 figures with captions")
        print("  • Tables: 10+ data tables")
        print("  • Patent Claims: 8 primary claims")
        
        print("\n" + "="*60)
        print(f"\nOpen the PDF: {os.path.abspath(pdf_path)}")
        
    else:
        print(f"\n✗ Status: FAILED")
        print(f"  PDF file was not created.")
        
# Verify the output
verify_output(OUTPUT_PDF)

## 10. Alternative: Generate PDF using markdown-pdf (Fallback)

In [ ]:
# Alternative method if WeasyPrint has issues
# This uses fpdf2 library as a fallback

def generate_simple_pdf_fallback():
    """Fallback PDF generation using fpdf2."""
    try:
        from fpdf import FPDF
        import textwrap
        
        print("Using fallback PDF generator (fpdf2)...")
        
        # Read markdown
        with open(MARKDOWN_FILE, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Create PDF
        pdf = FPDF()
        pdf.set_auto_page_break(auto=True, margin=15)
        pdf.add_page()
        
        # Add font
        pdf.set_font('Helvetica', size=10)
        
        # Process content line by line
        for line in content.split('\n'):
            # Handle headings
            if line.startswith('# '):
                pdf.set_font('Helvetica', 'B', 18)
                pdf.cell(0, 12, line[2:], ln=True)
                pdf.set_font('Helvetica', size=10)
            elif line.startswith('## '):
                pdf.set_font('Helvetica', 'B', 14)
                pdf.cell(0, 10, line[3:], ln=True)
                pdf.set_font('Helvetica', size=10)
            elif line.startswith('### '):
                pdf.set_font('Helvetica', 'B', 12)
                pdf.cell(0, 8, line[4:], ln=True)
                pdf.set_font('Helvetica', size=10)
            elif line.startswith('!['):
                # Skip image references (add placeholder)
                pdf.set_font('Helvetica', 'I', 9)
                pdf.cell(0, 6, '[Figure]', ln=True)
                pdf.set_font('Helvetica', size=10)
            elif line.strip():
                # Regular text - wrap long lines
                pdf.multi_cell(0, 5, line)
            else:
                pdf.ln(3)
        
        # Save
        fallback_output = 'IDF_Document_Simple.pdf'
        pdf.output(fallback_output)
        print(f"Fallback PDF saved as: {fallback_output}")
        return True
        
    except ImportError:
        print("fpdf2 not installed. Run: pip install fpdf2")
        return False
    except Exception as e:
        print(f"Fallback failed: {e}")
        return False

# Uncomment to use fallback if main method fails:
# generate_simple_pdf_fallback()

## Quick Reference

**Output Files:**
- `IDF_Document.pdf` - Main PDF output with full formatting
- `IDF_Document.html` - Intermediate HTML file

**Requirements:**
- Python 3.8+
- markdown
- weasyprint
- Pillow

**Note:** WeasyPrint may require GTK libraries on Windows. If you encounter issues, install GTK from: https://github.com/nicholasbishop/WeasyPrint-wheels/releases